# Conditional Probability: Does Bitcoin Rebound After a Down Day?

## Research question

> **If Bitcoin went down today, what is the probability that it goes up tomorrow?**

This notebook uses historical Bitcoin daily returns to introduce **conditional probability** through a real financial example.

The key idea is to compare two questions:

1. **Unconditional probability:** What is the probability that Bitcoin goes up tomorrow?
2. **Conditional probability:** What is the probability that Bitcoin goes up tomorrow **given that Bitcoin went down today**?

The second question is the important one because we are incorporating information about today's return.

### Probability notation

We will use:

- $A$ = Bitcoin goes **up tomorrow**
- $B$ = Bitcoin **went down today**

Our main quantity is therefore

$$
P(A\mid B)
$$

which reads:

> **The probability of A given B.**

**Reference:** Blitzstein & Hwang, *Introduction to Probability*, 2nd ed., Chapter 2, especially Sections 2.1–2.2.

## 1. The conceptual map

Before touching the data, define the events clearly.

| Symbol | Meaning in this notebook |
|---|---|
| $A$ | Tomorrow's Bitcoin return is positive |
| $A^c$ | Tomorrow's Bitcoin return is not positive |
| $B$ | Today's Bitcoin return is negative |
| $B^c$ | Today's Bitcoin return is not negative |
| $A\cap B$ | Today Bitcoin fell **and** tomorrow Bitcoin rises |
| $A\cup B$ | Today Bitcoin fell **or** tomorrow Bitcoin rises (or both) |
| $P(A)$ | Probability that Bitcoin rises |
| $P(B)$ | Probability that Bitcoin falls |
| $P(A\cap B)$ | Probability that both events occur |
| $P(A\mid B)$ | Probability Bitcoin rises given that it fell today |

### Important distinction

$A\cap B$ and $P(A\mid B)$ are **not the same thing**.

- $A\cap B$ means **A and B happen together**.
- $P(A\cap B)$ measures how frequently they happen together.
- $P(A\mid B)$ asks how likely A is **inside the subset where B happened**.

This distinction is one of the most important ideas in this notebook.

## 2. Why conditioning changes the question

Suppose our dataset contains 2,000 trading days.

If we ask $P(A)$, we look at **all 2,000 days** and calculate the fraction of days associated with an upward return.

But if we ask $P(A\mid B)$, we first restrict our attention to **only the days on which Bitcoin fell**. We then ask how often the following day was positive.

So conditional probability can be understood as:

$$
\boxed{\text{Conditioning = restricting the sample space using information we know.}}
$$

Blitzstein & Hwang emphasize this interpretation: conditional probability updates our uncertainty after observing evidence. They also stress that $P(\cdot\mid B)$ can be viewed as a new probability function that incorporates the information that $B$ occurred.

A useful mental model is:

**Unconditional:**

$$
P(A)=\frac{\text{number of observations where A occurs}}{\text{number of observations}}.
$$

**Conditional:**

$$
P(A\mid B)=\frac{\text{number of observations where A and B occur}}{\text{number of observations where B occurs}}.
$$

This is the empirical/frequentist version we will estimate from the Bitcoin sample.

## 3. Import libraries and download the data

In [ ]:
import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import kagglehub

warnings.filterwarnings("ignore")

# Download the historical Bitcoin dataset.
downloaded_path = kagglehub.dataset_download(
    "novandraanugrah/bitcoin-historical-datasets-2018-2024"
)


In [ ]:
def load_bitcoin_data(
    dir_path: str | Path = downloaded_path,
    tf: str = "1d"
) -> pd.DataFrame:
    """Load Bitcoin historical data for the selected timeframe."""

    filename = f"btc_{tf}_data_2018_to_2025.csv"
    file_path = os.path.join(dir_path, filename)

    print(f"Loading: {file_path}")

    return pd.read_csv(
        file_path,
        parse_dates=["Close time"]
    )


df = load_bitcoin_data(tf="1d")
df.head()

## 4. Define the random outcome we care about

We first calculate the daily simple return:

$$
R_t = \frac{P_t-P_{t-1}}{P_{t-1}}
$$

where $P_t$ is the Bitcoin closing price on day $t$.

Then we convert the continuous return into two events:

$$
\text{Up}_t = \{R_t>0\}
$$

$$
\text{Down}_t = \{R_t<0\}.
$$

This is a useful probability trick: instead of working directly with the numerical return, we create **events** that can occur or not occur.

In [ ]:
# Daily simple return.
df["return"] = df["Close"].pct_change()

# The first observation has no previous close, so its return is undefined.
df = df.dropna(subset=["return"]).copy()

# Define the events Up and Down.
df["up"] = df["return"] > 0
df["down"] = df["return"] < 0

df[["Close time", "Close", "return", "up", "down"]].head()

## 5. First question: what is $P(A)$?

Let

$$
A = \{\text{Bitcoin goes up}
\}.
$$

The empirical probability is estimated by the sample proportion:

$$
\hat P(A)
=
\frac{\#\{\text{observations where }A\text{ occurs}\}}{\#\{\text{observations}\}}.
$$

Because a Boolean Series contains `True` and `False`, pandas can calculate this directly with `.mean()`:

- `True` behaves like 1
- `False` behaves like 0

Therefore, the mean is exactly the proportion of `True` observations.

In [ ]:
# Empirical probability that a Bitcoin daily return is positive.
p_up = df["up"].mean()

print(f"P(A) = P(Bitcoin goes up) = {p_up:.3f} ({p_up:.1%})")

## 6. Introduce tomorrow's event

Our question is specifically about **tomorrow**.

If row $t$ represents today, then row $t+1$ represents tomorrow. Therefore we shift the `up` indicator one row upward:

```python
df["tomorrow_up"] = df["up"].shift(-1)
```

After this transformation:

- today's row contains tomorrow's `up` value;
- the final row has no tomorrow in the dataset and therefore becomes missing.

We remove that final incomplete observation before calculating probabilities.

In [ ]:
# A = tomorrow is an up day.
df["tomorrow_up"] = df["up"].shift(-1)

# The final row has no next-day observation.
df = df.dropna(subset=["tomorrow_up"]).copy()
df["tomorrow_up"] = df["tomorrow_up"].astype(bool)

df[["Close time", "up", "down", "tomorrow_up"]].head()

## 7. The main question: $P(A\mid B)$

Now define the events precisely:

$$
A = \{\text{tomorrow goes up}\}
$$

$$
B = \{\text{today goes down}\}.
$$

The question

$$
\boxed{P(A\mid B)}
$$

means:

> Among the days when Bitcoin fell today, how often did Bitcoin rise tomorrow?

### The definition

$$
\boxed{
P(A\mid B)=\frac{P(A\cap B)}{P(B)}
}
$$

The book's Chapter 2 recap gives this same definition. The crucial idea is that **the denominator changes**: we no longer divide by all observations; we divide by the observations in $B$.

A short quotation from Blitzstein & Hwang captures the interpretation:

> “The conditional probability of A given B is”

followed by the definition above. — *Introduction to Probability*, Chapter 2, Section 2.9.

Reference: fileciteturn1file6L382-L395

In [ ]:
# B = today is a down day.
today_down = df["down"]

# Direct empirical estimate of P(A | B).
p_up_given_down = df.loc[today_down, "tomorrow_up"].mean()

print(
    f"P(A | B) = P(tomorrow up | today down) "
    f"= {p_up_given_down:.3f} ({p_up_given_down:.1%})"
)

## 8. Why `.loc[today_down].mean()` works

This line is worth understanding rather than memorizing:

```python
df.loc[today_down, "tomorrow_up"].mean()
```

It performs the conditional probability calculation in two conceptual steps.

### Step 1 — Condition on $B$

```python
df.loc[today_down]
```

keeps **only observations where $B$ occurred**: days when Bitcoin fell.

### Step 2 — Measure $A$ inside that subset

```python
df.loc[today_down, "tomorrow_up"].mean()
```

calculates the fraction of those selected observations for which $A$ occurred.

So in plain English:

$$
P(A\mid B)
=
\frac{\text{A and B observations}}{\text{B observations}}.
$$

That is the core concept of conditional probability.

## 9. Deriving the same answer using $P(A\cap B)/P(B)$

Now we will deliberately **not** use the direct `.loc[...].mean()` approach.

Instead, we will use the mathematical definition:

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)}.
$$

### Step 1 — Find $P(A\cap B)$

$A\cap B$ means **both events occur**:

$$
A\cap B
=
\{\text{tomorrow goes up AND today goes down}\}.
$$

In Python, `&` represents the intersection of Boolean conditions:

```python
df["tomorrow_up"] & df["down"]
```

Reference: Blitzstein & Hwang introduce intersections as the event that both events occur. fileciteturn0file0L442-L460

In [ ]:
# A ∩ B = tomorrow goes up AND today goes down.
both = df["tomorrow_up"] & df["down"]

# Empirical probability of the intersection.
p_intersection = both.mean()

# Empirical probability of the conditioning event B.
p_down = df["down"].mean()

# Conditional probability from the definition.
p_formula = p_intersection / p_down

print(f"P(A ∩ B) = {p_intersection:.4f}")
print(f"P(B)     = {p_down:.4f}")
print(f"P(A|B)   = {p_intersection:.4f} / {p_down:.4f} = {p_formula:.4f}")

print(f"\nDirect calculation: {p_up_given_down:.4f}")
print(f"Formula calculation: {p_formula:.4f}")
print(f"Same result? {np.isclose(p_up_given_down, p_formula)}")

## 10. The most important numerical intuition

Suppose, for illustration, that:

- there are 1,000 total observations;
- 450 days satisfy $B$ (today went down);
- 230 observations satisfy $A\cap B$ (today went down and tomorrow went up).

Then:

$$
P(A\cap B)=\frac{230}{1000}=0.23
$$

and

$$
P(B)=\frac{450}{1000}=0.45.
$$

Therefore:

$$
P(A\mid B)=\frac{0.23}{0.45}\approx0.511.
$$

Notice what happened:

$$
\frac{230}{1000}
\quad\longrightarrow\quad
\frac{230}{450}.
$$

We changed the reference population from **all days** to **down days only**.

This is the cleanest way to understand the denominator of conditional probability.

## 11. Contingency table: see the probability structure

A contingency table lets us see the four combinations of today's and tomorrow's states.

| | Tomorrow up ($A$) | Tomorrow not up ($A^c$) |
|---|---:|---:|
| Today down ($B$) | $A\cap B$ | $A^c\cap B$ |
| Today not down ($B^c$) | $A\cap B^c$ | $A^c\cap B^c$ |

The cell $A\cap B$ is especially important because it appears in the numerator of:

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)}.
$$

The entire row corresponding to $B$ becomes the new conditional sample space.

In [ ]:
# Create readable labels for the two events.
df["today"] = np.where(df["down"], "Down", "Not down")
df["tomorrow"] = np.where(df["tomorrow_up"], "Up", "Not up")

contingency_table = pd.crosstab(
    df["today"],
    df["tomorrow"],
    margins=True
)

contingency_table

## 12. Conditional probability as a ratio inside a row

From the contingency table, we can calculate:

$$
P(A\mid B)
=
\frac{\text{count}(A\cap B)}{\text{count}(B)}.
$$

So the conditional probability is a **row proportion** when the conditioning event is represented by a row.

This is why contingency tables are so useful for learning conditional probability: they make the denominator visible.

In [ ]:
down_days = df["down"].sum()
down_then_up = (df["down"] & df["tomorrow_up"]).sum()

print(f"Number of down days: {down_days}")
print(f"Number of down-then-up days: {down_then_up}")
print(f"P(A|B) = {down_then_up} / {down_days} = {down_then_up / down_days:.4f}")

## 13. Compare the three probabilities

We can now compare:

### Baseline probability
$$
P(A)=P(\text{tomorrow up})
$$

This ignores today's return.

### Conditional on a down day
$$
P(A\mid B)=P(\text{tomorrow up}\mid\text{today down})
$$

This uses the information that today was a down day.

### Conditional on an up day
$$
P(A\mid B^c)=P(\text{tomorrow up}\mid\text{today up})
$$

This asks whether today's direction changes the probability of tomorrow's direction.

In [ ]:
# P(tomorrow up | today up)
p_up_given_up = df.loc[df["up"], "tomorrow_up"].mean()

print("=" * 60)
print("PROBABILITY COMPARISON")
print("=" * 60)
print(f"P(A)       = {p_up:.3f} ({p_up:.1%})")
print(f"P(A | B)   = {p_up_given_down:.3f} ({p_up_given_down:.1%})")
print(f"P(A | Bᶜ)  = {p_up_given_up:.3f} ({p_up_given_up:.1%})")

## 14. A common mistake: $P(A\mid B)\neq P(B\mid A)$

Be very careful with the order of the events.

$$
P(A\mid B)
\neq
P(B\mid A)
$$

In our problem:

$$
P(\text{tomorrow up}\mid\text{today down})
$$

is **not** the same question as:

$$
P(\text{today down}\mid\text{tomorrow up}).
$$

Blitzstein & Hwang explicitly warn that the order around the conditioning bar matters and discuss confusing these quantities as the prosecutor's fallacy. fileciteturn1file1L104-L125

## 15. The multiplication rule: connecting $P(A\mid B)$ and $P(A\cap B)$

Starting from

$$
P(A\mid B)=\frac{P(A\cap B)}{P(B)},
$$

multiply both sides by $P(B)$:

$$
\boxed{P(A\cap B)=P(B)P(A\mid B)}.
$$

The same intersection can also be written in the other direction:

$$
\boxed{P(A\cap B)=P(A)P(B\mid A)}.
$$

Therefore:

$$
P(B)P(A\mid B)=P(A)P(B\mid A).
$$

This is the bridge that leads directly to Bayes' rule.

Blitzstein & Hwang state the intersection identity as $P(A\cap B)=P(B)P(A\mid B)=P(A)P(B\mid A)$. fileciteturn1file0L20-L34

## 16. Bayes' rule — preview

From the previous identity:

$$
P(B)P(A\mid B)=P(A)P(B\mid A),
$$

divide by $P(B)$:

$$
\boxed{
P(A\mid B)=\frac{P(B\mid A)P(A)}{P(B)}
}.
$$

This is **Bayes' rule**.

For this notebook, the direct conditional-probability definition is enough. Bayes' rule becomes especially useful when $P(B\mid A)$ is easier to calculate than $P(A\mid B)$.

Blitzstein & Hwang describe Bayes' rule as a way to relate $P(A\mid B)$ to $P(B\mid A)$. fileciteturn1file2L137-L144

## 17. What does the result actually tell us?

Suppose the calculation gives:

$$
P(A)=0.52
$$

and

$$
P(A\mid B)=0.54.
$$

Then after observing a down day, the empirical probability of an up day increases from 52% to 54%.

But this **does not automatically mean we have a profitable trading strategy**.

A probability difference can be small, unstable, caused by sampling variation, or disappear after transaction costs and out-of-sample testing.

For this notebook, the goal is first to understand the probability concept correctly. Statistical inference and trading strategy evaluation come later.

## 18. Important terminology: prior vs. posterior

You wrote:

> prior probability of A and posterior probability of A

There is an important correction here.

**Prior** and **posterior** are specifically Bayesian terms. We should not automatically call $P(A)$ a prior and $P(A\mid B)$ a posterior just because conditioning appears.

For this notebook, the clean terminology is:

- $P(A)$ = **unconditional / marginal probability** of $A$;
- $P(A\mid B)$ = **conditional probability** of $A$ given $B$.

In a Bayesian analysis, $P(A)$ can serve as a prior distribution and $P(A\mid B)$ can be interpreted as a posterior distribution after observing evidence $B$, but that is a broader modeling framework.

For now, focus on the simpler idea:

$$
\boxed{\text{new information} \Rightarrow \text{updated probability}}
$$

The book makes a closely related conceptual point: conditional probability is a probability function that incorporates the information that the conditioning event occurred. fileciteturn1file4L262-L311

## 19. Independence — the next concept to investigate

The natural follow-up question is:

> Does knowing that Bitcoin fell today actually change the probability of Bitcoin rising tomorrow?

If knowing $B$ does **not** change the probability of $A$, then:

$$
P(A\mid B)=P(A).
$$

This is the basic conditional-probability view of **independence**.

If instead:

$$
P(A\mid B)\neq P(A),
$$

then the observed condition $B$ changes the probability of $A$ in the historical sample.

That difference is exactly what makes this Bitcoin example interesting from a quantitative perspective.

## 20. Final recap

### The three core quantities

$$
\boxed{P(A)}
$$
**How often does Bitcoin go up?**

$$
\boxed{P(A\cap B)}
$$
**How often does Bitcoin go up tomorrow AND fall today?**

$$
\boxed{P(A\mid B)=\frac{P(A\cap B)}{P(B)}}
$$
**Among the days when Bitcoin fell today, how often did it rise tomorrow?**

### The key mental model

Think of conditional probability as changing the population you are looking at:

$$
\text{All days}
\quad\xrightarrow{\text{condition on }B}\quad
\text{Only days where }B\text{ occurred}.
$$

Then calculate the probability of $A$ inside that restricted set.

This is the main idea to carry forward.

## 21. Exercises for yourself

Try answering these before writing more code.

1. What exactly is event $A$ in this notebook?
2. What exactly is event $B$?
3. In words, what does $A\cap B$ mean?
4. Why is $P(A\cap B)$ generally smaller than $P(A\mid B)$?
5. What does the denominator $P(B)$ represent?
6. Why does `.loc[df["down"]]` correspond to conditioning on $B$?
7. What is the difference between $P(A)$ and $P(A\mid B)$?
8. Is $P(A\mid B)$ the same as $P(B\mid A)$? Why not?
9. If $P(A\mid B)=P(A)$, what probabilistic concept should you think of?
10. If $P(A\mid B)>P(A)$ in this Bitcoin sample, what does that tell you — and what does it **not** tell you about profitability?

## References

Blitzstein, J. K., & Hwang, J. (2019). *Introduction to Probability* (2nd ed.). Chapman & Hall/CRC.

- Chapter 1: Probability and counting — sample spaces, events, intersections, and empirical counting intuition.
- Chapter 2.1: The importance of thinking conditionally.
- Chapter 2.2: Definition and intuition.
- Chapter 2.3: Bayes' rule and the law of total probability.
- Chapter 2.5: Independence of events.
- Chapter 2.9: Recap.

The authors explicitly emphasize learning probability through stories, pictures, intuition, and multiple solutions rather than memorizing disconnected formulas. That philosophy is used throughout this notebook.